In [8]:
%pip install streamlit pandas langchain-core langchain-ollama langgraph tabulate pandas-stubs

In [9]:
import subprocess
if subprocess.call(["which", "ollama"], stdout=subprocess.PIPE, stderr=subprocess.PIPE) != 0:
    !sudo apt update
    !sudo apt install -y pciutils zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Ollama already installed")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,469 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information...

In [10]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [11]:
!ollama pull "deepseek-r1:7b"
!ollama pull "qwen2.5-coder:latest"

In [12]:
!mkdir -p app/{core,agent,db}
!touch app/__init__.py
!touch app/{core,agent,db}/__init__.py

In [13]:
%%writefile ./app/core/config.py
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Database configurations
DB_PATH = os.getenv("ENERGY_DB_PATH", "data/energy_data.db")

# LLM Configurations
REASONING_MODEL = os.getenv("REASONING_MODEL", "deepseek-r1:7b")
CODER_MODEL = os.getenv("CODER_MODEL", "qwen2.5-coder:latest") # or codeqwen:latest

Writing ./app/core/config.py


In [14]:
%%writefile app/core/state.py
from typing import TypedDict, Literal, Optional, List, Dict, Any


class ActionClassification(TypedDict):
    intent: Literal["data_add", "data_modify", "query", "report", "irrelevant"]
    summary: str
    reasoning: str


class SourceClassification(TypedDict):
    source: Literal["outage_reports", "demand_reports", "both"]
    reasoning: str


class AgentActionState(TypedDict):
    user_request: str
    classification: Optional[ActionClassification]
    source_classification: Optional[SourceClassification]

    issue_search_results: Optional[List[str]]
    db_search_results: Optional[Dict[str, Any]]

    messages: List[str]
    drafted_response: Optional[str]

Writing app/core/state.py


In [15]:
%%writefile app/db/database.py
import os
import sqlite3
import pandas as pd
from app.core.config import DB_PATH

def setup_database(csv_path: str, db_path: str = DB_PATH):
    df = pd.read_csv(csv_path)
    df.dropna(inplace=True)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    conn = sqlite3.connect(db_path)
    df.to_sql('demand_reports', conn, if_exists='replace', index=False)
    schema = pd.io.sql.get_schema(df, 'demand_reports')
    conn.close()
    return schema

def get_dynamic_schema(db_path: str = DB_PATH) -> str:
    try:
        if not os.path.exists(db_path):
            return "No tables exist in the database yet."
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
        schemas = [row[0] for row in cursor.fetchall() if row[0] is not None]
        conn.close()
        return "\n\n".join(schemas) if schemas else "No tables exist in the database yet."
    except Exception as e:
        return f"Error reading schema: {e}"

Writing app/db/database.py


In [16]:
%%writefile app/agent/llm.py
from langchain_ollama import ChatOllama
from app.core.config import REASONING_MODEL, CODER_MODEL

reasoning_llm = ChatOllama(
    model=REASONING_MODEL,
    reasoning=True,
    temperature=0.5,
    num_ctx=8192
)

coder_llm = ChatOllama(
    model=CODER_MODEL,
    temperature=0.0,
    num_ctx=2048
)

Writing app/agent/llm.py


In [17]:
%%writefile app/agent/nodes.py
import re
import sqlite3
import pandas as pd
import logging
from typing import Literal
from langchain_core.prompts import PromptTemplate
from langgraph.types import Command

from app.core.state import AgentActionState, ActionClassification, SourceClassification
from app.core.config import DB_PATH
from app.db.database import get_dynamic_schema
from app.agent.llm import reasoning_llm, coder_llm

# Ensure your logger is set to at least INFO level in your main app config to see these.
# e.g., logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def classify_intent(state: AgentActionState) -> Command[Literal["classify_sources", "draft_response"]]:
    logger.info("=== NODE: classify_intent ===")
    logger.info(f"Incoming State User Request: {state.get('user_request')}")

    structured_text_llm = reasoning_llm.with_structured_output(ActionClassification)

    classification_prompt = f"""
        You are an intelligent routing agent for an Energy Management System.
        Analyze the user's message and classify its primary intent based on the following strict rules:

        INTENT CATEGORIES:
        - 'data_add': The user explicitly provides a file path (e.g., .csv) or asks to ingest new data.
        - 'data_modify': The user asks to UPDATE, DELETE, or correct existing grid/energy data.
        - 'query': The user asks a specific, targeted question (e.g., "What was the peak load yesterday?", "Did substation A go down?").
        - 'report': The user asks for a broader analysis, trend summary, or comprehensive overview.
        - 'irrelevant': The prompt has absolutely nothing to do with energy grids, power supply, outages, or data analysis.

        User's request: {state['user_request']}
        """

    logger.info(f"LLM Prompt (classify_intent):\n{classification_prompt}")
    classification = structured_text_llm.invoke(classification_prompt)
    logger.info(f"LLM Output (Classification): {classification}")

    goto = "draft_response" if classification["intent"] == "irrelevant" else "classify_sources"
    logger.info(f"Routing to: {goto}")
    return Command(goto=goto, update={"classification": classification})


def classify_sources(state: AgentActionState) -> Command[Literal["data_agent", "report_agent", "analysis_agent"]]:
    logger.info("=== NODE: classify_sources ===")
    logger.info(f"Current Intent: {state['classification'].get('intent')}")

    structured_text_llm = reasoning_llm.with_structured_output(SourceClassification)

    classifications_prompt = f"""
        You must determine the correct data source required to answer the user's energy query.

        SOURCES:
        1. "outage_reports": Unstructured text data.
           -> Select this if the user asks about: grid faults, equipment failures, blackouts, weather damage, repair logs.
        2. "demand_reports": Structured telemetry/time-series data.
           -> Select this if the user asks about: MW/h consumption, peak load, generation capacity, voltage levels.
        3. "both":
           -> Select this if the query requires correlating an event with data.

        User's query: {state['user_request']}
        Intent Summary: {state["classification"].get("summary", "Unknown")}
        """

    logger.info(f"LLM Prompt (classify_sources):\n{classifications_prompt}")
    classification = structured_text_llm.invoke(classifications_prompt)
    logger.info(f"LLM Output (Source Classification): {classification}")

    intent = state["classification"]["intent"]

    if intent in ["data_add", "data_modify", "query", "report"]:
        goto = "data_agent"
    else:
        goto = "report_agent"

    logger.info(f"Routing to: {goto}")
    return Command(goto=goto, update={"source_classification": classification})


def data_agent(state: AgentActionState) -> Command[Literal["analysis_agent", "report_agent"]]:
    logger.info("=== NODE: data_agent ===")
    intent = state["classification"]["intent"]
    user_request = state["user_request"]
    updates = {}

    if intent == "data_add":
        match = re.search(r'([\w\-./\\]+\.csv)', user_request)
        if match:
            csv_file = match.group(1)
            logger.info(f"Detected CSV file for ingestion: {csv_file}")
            try:
                df = pd.read_csv(csv_file)
                columns = ", ".join(df.columns)
                data_sample = df.head(1).to_markdown(index=False)

                naming_prompt = PromptTemplate.from_template("""
                    You are a database architect for a power grid operator.
                    Analyze the following CSV columns and sample data.
                    Columns: {columns}
                    Data Sample:
                    {sample}

                    Suggest a single, highly descriptive database table name.
                    - Use ONLY lowercase letters and underscores.
                    - Prefer industry-standard terms (e.g., `smart_meter_readings`, `substation_outages`, `grid_demand_hourly`).

                    CRITICAL: Output ONLY the table name, nothing else.
                    """)

                logger.info(f"LLM Prompt (Table Naming):\n{naming_prompt.format(columns=columns, sample=data_sample)}")
                chain = naming_prompt | reasoning_llm
                raw_name = chain.invoke({"columns": columns, "sample": data_sample}).content.strip()
                logger.info(f"LLM Raw Output (Table Name): {raw_name}")

                table_name = re.sub(r'[^a-z0-9_]', '', raw_name.lower()) or "unnamed_dataset"
                logger.info(f"Sanitized Table Name: {table_name}")

                conn = sqlite3.connect(DB_PATH)
                df.to_sql(table_name, conn, if_exists="replace", index=False)
                conn.close()

                msg = f"Data Agent: Analyzed `{csv_file}`. Automatically created table: `{table_name}` ({len(df)} rows)."
                updates["messages"] = state.get("messages", []) + [msg]
            except Exception as e:
                error_msg = f"Data Agent Error: Could not load {csv_file}. {e}"
                logger.error(error_msg)
                updates["messages"] = state.get("messages", []) + [error_msg]
        else:
            logger.warning("Intent was 'data_add' but no .csv file was found in the prompt.")
            updates["messages"] = state.get("messages", []) + [
                "Data Agent: Please provide a valid .csv filename to import."]

        return Command(goto="report_agent", update=updates)

    elif intent in ["query", "data_modify", "report"]:
        current_schema = get_dynamic_schema(DB_PATH)
        action_type = "read-only SELECT query" if intent in ["query",
                                                             "report"] else "Data Manipulation Language (UPDATE/DELETE) query"

        sql_prompt = PromptTemplate.from_template("""
                    You are a Principal Data Engineer managing a SQLite database for a major energy grid.
                    Your task is to translate a human user's request into a precise, flawless {action_type}.

                    CURRENT DATABASE SCHEMA:
                    {schema}

                    CRITICAL SCHEMA RULES (ANTI-HALLUCINATION & ANTI-BLEED):
                    1. NEVER invent, guess, or assume table or column names.
                    2. You MUST ONLY use the exact tables and columns explicitly listed in the CURRENT DATABASE SCHEMA above.
                    3. STRICT TABLE-COLUMN MATCHING: You must verify that the columns you use actually belong to the table you are querying in the FROM clause. Do not use a column from Table A in a query for Table B.
                    4. If the user asks for a category (e.g., "region") that does not exist in the target table, look for the closest geographical equivalent that DOES exist in that specific table (e.g., use "state" or "county" instead).

                    VOCABULARY MAPPING INSTRUCTIONS (Human to SQL):
                    - Apply these conceptual mappings ONLY IF the target columns actually exist in the specific table you are querying:
                    - "blackout" / "power cut" -> look for columns related to `outages`, `status`, or `faults`.
                    - "usage" / "load" / "power draw" -> look for `demand`, `consumption`, or `mw` columns.
                    - "location" / "substation" / "plant" / "region" -> look for `state`, `county`, `location_id`, `facility`, or `site` columns.
                    - Use `LOWER(column_name) LIKE '%term%'` for text searches to handle case-insensitivity.

                    SQLITE DIALECT RULES:
                    - SQLite does not have advanced date types. Use `date(column)`, `datetime(column)`, or `strftime()` for time filtering.
                    - Always use double quotes for column names if they contain spaces or special characters.
                    - If this is a SELECT query, add a `LIMIT 500` at the end to prevent returning excessively large datasets, unless the user explicitly asks for a count or aggregation (SUM, AVG).

                    User Request: {question}

                    Return ONLY the raw SQL query. Do not include markdown blocks, explanations, or comments.
                    SQL Query:
                    """)

        formatted_sql_prompt = sql_prompt.format(action_type=action_type, schema=current_schema, question=user_request)
        logger.info(f"LLM Prompt (SQL Generation):\n{formatted_sql_prompt}")

        chain = sql_prompt | coder_llm
        raw_sql_response = chain.invoke(
            {"action_type": action_type, "schema": current_schema, "question": user_request}).content.strip()

        logger.info(f"LLM Raw Output (SQL): {raw_sql_response}")
        sql_query = re.sub(r"```[sS][qQ][lL]?", "", raw_sql_response).replace("```", "").strip()
        logger.info(f"Sanitized SQL Query to execute: {sql_query}")

        updates["messages"] = state.get("messages", []) + [f"Data Agent generated SQL: {sql_query}"]

        try:
            conn = sqlite3.connect(DB_PATH)
            if intent in ["query", "report"]:
                logger.info("Executing SELECT query...")
                results_df = pd.read_sql_query(sql_query, conn)
                logger.info(f"Query returned {len(results_df)} rows.")

                results_df = results_df.astype(str)
                updates["db_search_results"] = results_df.to_dict(orient='records')
                next_node = "analysis_agent"
            else:
                logger.info("Executing DML query...")
                cursor = conn.cursor()
                cursor.execute(sql_query)
                conn.commit()
                logger.info(f"DML executed. {cursor.rowcount} row(s) affected.")
                updates["messages"].append(
                    f"Data Agent: Database updated successfully. {cursor.rowcount} row(s) affected.")
                next_node = "report_agent"
            conn.close()
        except Exception as e:
            logger.error(f"SQL Execution Failed: {str(e)}")
            updates["messages"].append(f"Data Agent Error: SQL Execution Failed: {str(e)}")
            next_node = "report_agent"

        logger.info(f"Routing to: {next_node}")
        return Command(goto=next_node, update=updates)

    logger.warning("Fell through data_agent logic. Routing to report_agent by default.")
    return Command(goto="report_agent", update=updates)


def analysis_agent(state: AgentActionState) -> Command[Literal["report_agent"]]:
    logger.info("=== NODE: analysis_agent ===")

    db_results = state.get('db_search_results', 'None')
    # Truncate db_results in logs if it's massive to avoid blowing up the console
    db_results_log = str(db_results)[:1000] + "... [TRUNCATED]" if len(str(db_results)) > 1000 else str(db_results)

    analysis_prompt = f"""
    You are an Expert Energy Data Analyst. Your job is to interpret raw database returns and unstructured logs, translating them into meaningful insights.

    User Query: {state['user_request']}

    Database Results (JSON Format): {db_results}
    Unstructured Data Logs: {state.get('issue_search_results', 'None')}

    INSTRUCTIONS:
    1. If the database results are empty or "None", state explicitly: "No data was found matching the parameters of the query."
    2. Do not just list the data. Extract key metrics (e.g., peak demand, average duration of outages, most affected substations).
    3. Identify any obvious anomalies or trends (e.g., "Demand spiked at 18:00").
    4. Keep your analysis objective, data-driven, and focused strictly on answering the User Query.

    Draft your analytical findings below:
    """

    logger.info(f"LLM Prompt (Analysis):\n[Prompt includes DB Results: {db_results_log}]\n...")
    response = reasoning_llm.invoke(analysis_prompt)
    logger.info(f"LLM Output (Analysis):\n{response.content}")

    messages = state.get("messages", []) + [f"Analysis Agent: {response.content}"]
    return Command(goto="report_agent", update={"messages": messages})


def report_agent(state: AgentActionState) -> Command[Literal["draft_response"]]:
    logger.info("=== NODE: report_agent ===")

    report_prompt = f"""
    You are the Lead Communications Officer for a Power Grid Authority.
    Take the raw agent logs and data analysis provided below and draft a clear, professional, and concise response to the user.

    User's Original Request: {state['user_request']}

    Agent Logs & Data Analysis:
    {state.get('messages', ['No analysis available.'])}

    FORMATTING RULES:
    - If the user asked a simple query, provide a direct, concise answer.
    - If the user asked for a report, format the response with professional headers (e.g., **Executive Summary**, **Key Metrics**, **Details**).
    - Exclude internal system jargon (e.g., do not say "The SQL query returned...", instead say "The grid data indicates...").
    - Ensure the tone is authoritative, reassuring, and clear.
    """

    logger.info(f"LLM Prompt (Report Generation):\n{report_prompt}")
    response = reasoning_llm.invoke(report_prompt)
    logger.info(f"LLM Output (Report Draft):\n{response.content}")

    return Command(goto="draft_response", update={"drafted_response": response.content})


def draft_response(state: AgentActionState):
    logger.info("=== NODE: draft_response ===")

    if state.get("classification", {}).get("intent") == "irrelevant":
        logger.info("Intent was irrelevant. Bypassing drafted response for default fallback.")
        return {
            "drafted_response": "I am an energy management assistant. I can only assist with power supply, grid demand, outage reporting, and related data queries. How can I help you with your energy infrastructure today?"}

    logger.info("Final response ready for delivery.")
    return {"drafted_response": state["drafted_response"]}

Writing app/agent/nodes.py


In [18]:
%%writefile app/agent/graph.py
from langgraph.constants import START, END
from langgraph.graph import StateGraph

from app.core.state import AgentActionState
from app.agent.nodes import classify_intent, classify_sources, data_agent, analysis_agent, report_agent, draft_response


def build_graph():
    builder = StateGraph(AgentActionState)
    builder.add_node("classify_intent", classify_intent)
    builder.add_node("classify_sources", classify_sources)
    builder.add_node("data_agent", data_agent)
    builder.add_node("analysis_agent", analysis_agent)
    builder.add_node("report_agent", report_agent)
    builder.add_node("draft_response", draft_response)

    builder.add_edge(START, "classify_intent")
    builder.add_edge("draft_response", END)

    return builder.compile()


app = build_graph()

Writing app/agent/graph.py


In [19]:
!pip install pyngrok

In [20]:
import os
from pyngrok import ngrok

if "NGROK_AUTH_TOKEN" not in os.environ:
    os.environ["NGROK_AUTH_TOKEN"] = input("Enter your token: ")

ngrok.set_auth_token(os.environ.get("NGROK_AUTH_TOKEN"))

In [21]:
tunnel = ngrok.connect(8501)
print(tunnel.public_url)

https://077b-34-125-59-220.ngrok-free.app


In [22]:
%%writefile app.py
import time

import streamlit as st
import os
import sqlite3
import pandas as pd

from app.agent.graph import app as agent_app
from app.core.config import DB_PATH, DATA_DIR

def stream_response(text):
    """Generator to simulate streaming text output."""
    for word in text.split(" "):
        yield word + " "
        time.sleep(0.02)

st.set_page_config(
    page_title="Energy Management Assistant",
    layout="wide"
)

st.title("Agent-Based Energy Management Assistant")

# --- UI Layout: Sidebar ---
with st.sidebar:
    st.header("Data Management")
    st.write("Upload CSVs.")

    uploaded_file = st.file_uploader("Upload CSV", type=["csv"])

    if uploaded_file is not None:
        file_path = os.path.join(DATA_DIR, uploaded_file.name)
        with open(file_path, "wb") as f:
            f.write(uploaded_file.getbuffer())
        st.success(f"Saved `{file_path}` locally.")

        if st.button("Import Data to Database"):
            with st.spinner(f"Instructing Data Agent to import {file_path}..."):
                initial_state = {
                    "user_request": f"Import the data from {file_path}",
                    "messages": []
                }
                try:
                    result = agent_app.invoke(initial_state)
                    st.info(result["drafted_response"])
                except Exception as e:
                    st.error(f"Import failed: {e}")

    st.markdown("---")
    st.subheader("Sample Queries:")
    st.code('"Summarize outage_reports by region"')
    st.code('"What is the total demand in the demand_reports table?"')
    st.code('"Update the status in outage_reports to Resolved where region is North"')

# --- UI Layout: Main Tabs ---
tab1, tab2 = st.tabs(["Assistant Chat", "Database Viewer"])

# --- TAB 1: Chat Interface ---
with tab1:
    # 1. Create a fixed-height, scrollable container for the chat history
    chat_container = st.container(height=600)

    if "messages" not in st.session_state:
        st.session_state.messages = []

    # 2. Render all existing messages INSIDE the scrollable container
        # 2. Render all existing messages INSIDE the scrollable container
        with chat_container:
            for message in st.session_state.messages:
                with st.chat_message(message["role"]):
                    st.markdown(message["content"])
                    # Persist the logs/thoughts in an expander for past messages
                    if message.get("logs"):
                        with st.expander("🧠 View Agent Internal Execution Logs"):
                            for log in message["logs"]:
                                st.info(log)

    # 3. The chat input natively sticks to the bottom of the active tab/screen
    if prompt := st.chat_input("Ask about energy demand, supply, or outages..."):
        st.session_state.messages.append({"role": "user", "content": prompt})

        # 4. Ensure new messages are also written INSIDE the container
        with chat_container:
            with st.chat_message("user"):
                st.markdown(prompt)

            with st.chat_message("assistant"):
                agent_logs = []
                final_state = None

                try:
                    initial_state = {
                        "user_request": prompt,
                        "messages": []
                    }

                    # Use st.status for interactive, real-time progress tracking
                    with st.status("Initializing agents...", expanded=True) as status:
                        # Stream through the LangGraph nodes instead of invoking all at once
                        for event in agent_app.stream(initial_state):
                            for node_name, node_state in event.items():
                                # Dynamically update the UI text to show which agent is working
                                status.update(label=f"⚙️ Agent `{node_name}` is working...", state="running")
                                final_state = node_state  # Keep track of the latest state

                                # Optionally write to the status box so users see the step history
                                st.write(f"Completed step: **{node_name}**")

                        # Mark as complete and collapse the status box
                        status.update(label="✅ Analysis complete!", state="complete", expanded=False)

                    # Process the final state after the graph finishes
                    if final_state:
                        raw_response = final_state.get("drafted_response", "Error: No response drafted.")

                        # Extract DeepSeek's <think> tags to display them in the logs
                        if "<think>" in raw_response:
                            parts = raw_response.split("</think>")
                            think_content = parts[0].replace("<think>", "").strip()
                            response = parts[-1].strip()
                            if think_content:
                                agent_logs.append(f"DeepSeek Reasoning:\n{think_content}")
                        else:
                            response = raw_response

                        # Gather other internal graph logs
                        if final_state.get("messages"):
                            agent_logs.extend(final_state["messages"])
                    else:
                        response = "Error: Graph execution failed to return a state."

                except Exception as e:
                    response = f"An error occurred during execution: {e}"
                    st.error(response)

                # Stream the final response to the UI
                if "response" in locals():
                    st.write_stream(stream_response(response))

                # Show internal logs cleanly in an expander for the current turn
                if agent_logs:
                    with st.expander("🧠 View Agent Internal Execution Logs"):
                        for log in agent_logs:
                            st.info(log)

                # Save the assistant's response AND logs to state so they persist on reload
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": response if "response" in locals() else "Error",
                    "logs": agent_logs
                })
# --- TAB 2: Database Viewer ---
with tab2:
    st.header("Database Tables")
    db_path = DB_PATH

    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall()]

        if not tables:
            st.info("No tables found in the database. Upload a CSV in the sidebar to get started.")
        else:
            selected_table = st.selectbox("Select a table to view:", tables)

            try:
                df = pd.read_sql_query(f"SELECT * FROM {selected_table}", conn)
                # FIX APPLIED HERE: Replaced use_container_width with width='stretch'
                st.dataframe(df, width='stretch')
                st.caption(f"Showing {len(df)} rows from `{selected_table}`.")
            except Exception as e:
                st.error(f"Could not load table: {e}")

        conn.close()
    else:
        st.warning("Database has not been created yet. Upload and import a file first.")

Writing app.py


In [23]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.59.220:8501

INFO - === NODE: classify_intent ===
INFO - Incoming State User Request: hi
INFO - LLM Prompt (classify_intent):

        You are an intelligent routing agent for an Energy Management System.
        Analyze the user's message and classify its primary intent based on the following strict rules:

        INTENT CATEGORIES:
        - 'data_add': The user explicitly provides a file path (e.g., .csv) or asks to ingest new data.
        - 'data_modify': The user asks to UPDATE, DELETE, or correct existing grid/energy data.
        - 'query': The user asks a specific, targeted question (e.g., "What was the peak load yesterday?", "Did substation A go down?").
        - 'report': The user asks for a broader analysis, trend summary, or comprehensive overview.
        - 'irrelevant': The prompt has absolutely nothing to do

  Stopping...
